# 01 — Tiền xử lý Dữ liệu (Preprocessing)

## Mục tiêu
- Load toàn bộ 14 file CSV từ `data/raw/`
- Kiểm tra kiểu dữ liệu, giá trị null, duplicate
- Join các bảng liên quan (orders ↔ customers ↔ geography ↔ payments…)
- Export file đã làm sạch sang `data/processed/`

## Input
`data/raw/*.csv`

## Output
`data/processed/merged_orders.parquet`

## Người phụ trách
_(Điền tên thành viên)_

## 1. Basic Setting

In [2]:

%load_ext autoreload
%autoreload 2

In [3]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd


warnings.filterwarnings("ignore")
pd.options.display.max_rows = 50
pd.options.display.max_columns = None

In [4]:
import os
import sys
from pathlib import Path

ROOT = Path("D:/VinAI")
src_path = str(ROOT / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(src_path)              
print(src_path in sys.path)  

D:\VinAI\src
True


In [ ]:
import importlib
import data_loader.loader as loader
importlib.reload(loader)
from data_loader.loader import save, load_all

In [6]:
RAW       = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

In [7]:

import shutil
import kagglehub

kaggle_path = Path(kagglehub.competition_download("datathon-2026-round-1"))

for f in kaggle_path.glob("*.csv"):
    dest = RAW / f.name
    shutil.copy(f, dest)
    print(f"✓ {f.name}")

✓ customers.csv
✓ geography.csv
✓ inventory.csv
✓ orders.csv
✓ order_items.csv
✓ payments.csv
✓ products.csv
✓ promotions.csv
✓ returns.csv
✓ reviews.csv
✓ sales.csv
✓ sample_submission.csv
✓ shipments.csv
✓ web_traffic.csv


In [8]:
tables = load_all(RAW)

for name, df in tables.items():
    print(f"{name:15s} {df.shape}")

products        (2412, 8)
customers       (121930, 7)
promotions      (50, 10)
geography       (39948, 4)
orders          (646945, 8)
order_items     (714669, 7)
payments        (646945, 4)
shipments       (566067, 4)
returns         (39939, 7)
reviews         (113551, 7)
sales           (3833, 3)
inventory       (60247, 17)
web_traffic     (3652, 7)


## Load and clean data

### Handle Table

In [36]:
DATE_COLS = {
    "orders":     ["order_date"],
    "customers":  ["signup_date"],
    "shipments":  ["ship_date", "delivery_date"],
    "returns":    ["return_date"],
    "reviews":    ["review_date"],
    "sales":      ["Date"],
    "inventory":  ["snapshot_date"],
    "web_traffic":["date"],
    "promotions": ["start_date", "end_date"],
}

for tbl, cols in DATE_COLS.items():
    for col in cols:
        tables[tbl][col] = pd.to_datetime(tables[tbl][col])

In [38]:
for name, df in tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if not nulls.empty:
        print(f"\n--- {name} ---")
        print(nulls)


--- promotions ---
applicable_category    40
dtype: int64

--- order_items ---
promo_id      438353
promo_id_2    714463
dtype: int64


In [11]:
for name, df in tables.items():
    n_dup = df.duplicated().sum()
    if n_dup > 0:
        print(f"{name}: {n_dup} duplicates")

### Handle each table

In [12]:
for name in tables:
    print(f"Processing {name}...")
    before = len(tables[name])
    tables[name] = tables[name].drop_duplicates()
    after = len(tables[name])
    if before != after:
        print(f"{name}: dropped {before - after} rows")
    else:
        print(f"{name}: no duplicates found")

Processing products...
products: no duplicates found
Processing customers...
customers: no duplicates found
Processing promotions...
promotions: no duplicates found
Processing geography...
geography: no duplicates found
Processing orders...
orders: no duplicates found
Processing order_items...
order_items: no duplicates found
Processing payments...
payments: no duplicates found
Processing shipments...
shipments: no duplicates found
Processing returns...
returns: no duplicates found
Processing reviews...
reviews: no duplicates found
Processing sales...
sales: no duplicates found
Processing inventory...
inventory: no duplicates found
Processing web_traffic...
web_traffic: no duplicates found


In [13]:
tables["shipments"]["shipping_fee"] = tables["shipments"]["shipping_fee"].fillna(0)

In [14]:

orders_with_discount_df = tables["order_items"][
    tables["order_items"]["promo_id"].notna()
]

print(orders_with_discount_df)

print(f"Có KM: {len(orders_with_discount_df) / len(tables['order_items']) * 100:.1f}%")

        order_id  product_id  quantity  unit_price  discount_amount  \
41316      46253        2250         6     1253.04          1127.74   
41317      46254        2251         7     1249.71          1312.20   
41319      46257         785         3      598.98           269.54   
41320      46257         786         2      611.66           183.50   
41321      46258        1093         6      604.71           544.24   
...          ...         ...       ...         ...              ...   
714648    834325         428         1    11424.45          2284.89   
714649    834326         927         6     4823.52          5788.22   
714650    834329         736         3     3701.16          2220.70   
714651    834333         505         6    19396.70         23276.04   
714652    834335         668         7    13128.07         18379.30   

          promo_id promo_id_2  
41316   PROMO-0006        NaN  
41317   PROMO-0006        NaN  
41319   PROMO-0006        NaN  
41320   PROMO-0006 

In [39]:

invalid = tables["products"][tables["products"]["cogs"] >= tables["products"]["price"]]
print(f"products vi phạm cogs >= price: {len(invalid)}")

print(tables["sales"][tables["sales"]["Revenue"] < 0].shape)
print(tables["sales"][tables["sales"]["COGS"] < 0].shape)

products vi phạm cogs >= price: 0
(0, 3)
(0, 3)


## Divide dim fact

In [43]:
#load raw data
products = tables['products'].copy()
orders   = tables['orders'].copy()
sales    = tables['sales'].copy()
order_items = tables['order_items'].copy()
payments = tables['payments'].copy()
customers = tables['customers'].copy()
geography = tables['geography'].copy()
promotions = tables['promotions'].copy()
shipments = tables['shipments'].copy()
returns = tables['returns'].copy()
reviews = tables['reviews'].copy()
inventory = tables['inventory'].copy()
web_traffic = tables['web_traffic'].copy()



In [45]:
# dim product
dim_products = products.copy()
dim_products['gross_margin_pct'] = ((dim_products['price'] - dim_products['cogs']) / dim_products['price'] * 100).round(2) # cứ bao nhiêu thu về thì lời bao nhiêu phần trăm
dim_products['profit_per_unit']  = (dim_products['price'] - dim_products['cogs']).round(2) ## Mỗi cái bán ra, lời bao nhiêu
print(dim_products.head())

avg_rating = reviews.groupby('product_id')['rating'].agg(['mean', 'count']).reset_index()
avg_rating.columns = ['product_id', 'avg_rating', 'review_count']
avg_rating['avg_rating'] = avg_rating['avg_rating'].round(2)

ret_stats = returns.groupby('product_id').agg(
    total_returns=('return_id', 'count'),
    total_return_qty=('return_quantity', 'sum'),
    total_refund=('refund_amount', 'sum')
).reset_index()

dim_products = dim_products.merge(avg_rating, on='product_id', how='left')
dim_products = dim_products.merge(ret_stats, on='product_id', how='left')
dim_products[['review_count', 'total_returns', 'total_return_qty']] = \
    dim_products[['review_count', 'total_returns', 'total_return_qty']].fillna(0).astype(int)
dim_products['total_refund'] = dim_products['total_refund'].fillna(0)

save(dim_products, 'dim_products.csv')

   product_id      product_name    category   segment size   color  \
0         536  SaigonFlex UC-01  Streetwear  Everyday    S   green   
1         537  SaigonFlex UC-02  Streetwear  Everyday    M  silver   
2         538  SaigonFlex UC-03  Streetwear  Everyday    L    pink   
3         539  SaigonFlex UC-04  Streetwear  Everyday   XL  yellow   
4         540  SaigonFlex UC-05  Streetwear  Everyday    S     red   

          price          cogs  gross_margin_pct  profit_per_unit  
0  11059.650000   9704.842875             12.25          1354.81  
1   9523.076013   5393.870254             43.36          4129.21  
2  15951.633158  11371.919278             28.71          4579.71  
3  15753.717299   8573.172954             45.58          7180.54  
4  15766.334536  14063.570406             10.80          1702.76  
Đã lưu: data/processed/dim_products.csv.csv


In [18]:
save(geography, 'dim_geography.csv')

Đã lưu: data/processed/dim_geography.csv.csv


In [19]:
save(promotions, 'dim_promotions.csv')

Đã lưu: data/processed/dim_promotions.csv.csv


In [47]:
# Denormalized fact table
fact = order_items.copy()

# Join orders
fact = fact.merge(
    orders[['order_id', 'order_date', 'customer_id', 'zip',
            'order_status', 'payment_method', 'device_type', 'order_source']],
    on='order_id', how='left'
)

# Join products (key fields only, to avoid bloat)
fact = fact.merge(
    products[['product_id', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs']],
    on='product_id', how='left'
)

#  Join payments
fact = fact.merge(
    payments[['order_id', 'payment_value', 'installments']],
    on='order_id', how='left'
)

#  Join geography (region, city)
fact = fact.merge(
    geography[['zip', 'city', 'region']].drop_duplicates(subset='zip'),
    on='zip', how='left'
)

# Computed fields
fact['line_revenue']       = (fact['quantity'] * fact['unit_price']).round(2)
fact['line_cost']          = (fact['quantity'] * fact['cogs']).round(2)
fact['line_gross_profit']  = (fact['line_revenue'] - fact['line_cost']).round(2)
fact['line_margin_pct']    = np.where(
    fact['line_revenue'] > 0,
    (fact['line_gross_profit'] / fact['line_revenue'] * 100).round(2),
    0
)
fact['has_promo']          = (~fact['promo_id'].isna() & (fact['promo_id'] != '')).astype(int)
fact['has_double_promo']   = (~fact['promo_id_2'].isna() & (fact['promo_id_2'] != '')).astype(int)
fact['discount_pct']       = np.where(
    fact['line_revenue'] + fact['discount_amount'] > 0,
    (fact['discount_amount'] / (fact['line_revenue'] + fact['discount_amount']) * 100).round(2),
    0
)

# Date dimensions
fact['order_year']    = fact['order_date'].dt.year
fact['order_month']   = fact['order_date'].dt.month
fact['order_quarter'] = fact['order_date'].dt.quarter
fact['order_dow']     = fact['order_date'].dt.dayofweek  
fact['order_dow_name']= fact['order_date'].dt.day_name()
fact['order_ym']      = fact['order_date'].dt.to_period('M').astype(str)

save(fact, 'fact_orders_enriched.csv')

Đã lưu: data/processed/fact_orders_enriched.csv.csv


In [49]:
fact_ret = returns.copy()
fact_ret = fact_ret.merge(
    products[['product_id', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs']],
    on='product_id', how='left'
)
fact_ret = fact_ret.merge(
    orders[['order_id', 'order_date', 'customer_id', 'order_source', 'device_type']],
    on='order_id', how='left'
)
# Time to return
fact_ret['days_to_return'] = (fact_ret['return_date'] - fact_ret['order_date']).dt.days
fact_ret['return_year']    = fact_ret['return_date'].dt.year
fact_ret['return_month']   = fact_ret['return_date'].dt.month
fact_ret['return_ym']      = fact_ret['return_date'].dt.to_period('M').astype(str)

save(fact_ret, 'fact_returns_enriched.csv')

Đã lưu: data/processed/fact_returns_enriched.csv.csv


In [51]:
#RFM Segmentation phan bo khach hang theo 3 tieu chi: Recency (R), Frequency (F), Monetary (M)
ref_date = orders['order_date'].max() + pd.Timedelta(days=1)
print(f"  Reference date for Recency: {ref_date.date()}")

# Compute RFM per customer
rfm = orders.merge(payments[['order_id', 'payment_value']], on='order_id', how='left')
rfm = rfm[rfm['order_status'] != 'cancelled']  # Exclude cancelled

rfm_agg = rfm.groupby('customer_id').agg(
    last_order_date   = ('order_date', 'max'),
    first_order_date  = ('order_date', 'min'),
    frequency         = ('order_id', 'nunique'),
    monetary          = ('payment_value', 'sum'),
    avg_order_value   = ('payment_value', 'mean'),
).reset_index()

rfm_agg['recency_days'] = (ref_date - rfm_agg['last_order_date']).dt.days
rfm_agg['tenure_days']  = (ref_date - rfm_agg['first_order_date']).dt.days
rfm_agg['monetary']     = rfm_agg['monetary'].round(2)
rfm_agg['avg_order_value'] = rfm_agg['avg_order_value'].round(2)

#  RFM Scores (quintiles 1-5)
rfm_agg['R_score'] = pd.qcut(rfm_agg['recency_days'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm_agg['F_score'] = pd.qcut(rfm_agg['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm_agg['M_score'] = pd.qcut(rfm_agg['monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm_agg['RFM_score'] = rfm_agg['R_score'] * 100 + rfm_agg['F_score'] * 10 + rfm_agg['M_score']

#  RFM Segment Labels
def rfm_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r >= 3 and f >= 1 and m >= 3:
        return 'Potential Loyalists'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f >= 4 and m >= 4:
        return 'Cant Lose Them'
    elif r <= 2 and f <= 2:
        return 'Lost'
    elif r == 3 and f <= 2:
        return 'About To Sleep'
    else:
        return 'Need Attention'

rfm_agg['rfm_segment'] = rfm_agg.apply(rfm_segment, axis=1)

#  Merge with customer demographics
dim_cust = customers.merge(rfm_agg, on='customer_id', how='left')
dim_cust = dim_cust.merge(
    geography[['zip', 'region']].drop_duplicates(subset='zip'),
    on='zip', how='left'
)

# Signup year/month
dim_cust['signup_year']  = dim_cust['signup_date'].dt.year
dim_cust['signup_month'] = dim_cust['signup_date'].dt.month
dim_cust['signup_ym']    = dim_cust['signup_date'].dt.to_period('M').astype(str)

# Fill NAs for customers without orders
dim_cust['frequency']    = dim_cust['frequency'].fillna(0).astype(int)
dim_cust['monetary']     = dim_cust['monetary'].fillna(0)
dim_cust['rfm_segment']  = dim_cust['rfm_segment'].fillna('Never Purchased')

# Segment distribution
print("\n  RFM Segment distribution:")
print(dim_cust['rfm_segment'].value_counts().to_string())

save(dim_cust, 'dim_customers_rfm.csv')

  Reference date for Recency: 2023-01-01

  RFM Segment distribution:
rfm_segment
Never Purchased        33807
Lost                   24996
Champions              24490
Loyal Customers        18163
At Risk                10221
New Customers           4865
About To Sleep          4222
Potential Loyalists     1166
Đã lưu: data/processed/dim_customers_rfm.csv.csv


In [53]:
#fact shipment for logistics analysis
fact_ship = shipments.copy()
fact_ship['delivery_days']     = (fact_ship['delivery_date'] - fact_ship['ship_date']).dt.days
fact_ship['processing_days']   = None  

# Join order info for processing time
fact_ship = fact_ship.merge(
    orders[['order_id', 'order_date', 'zip', 'order_status']],
    on='order_id', how='left'
)
fact_ship['processing_days'] = (fact_ship['ship_date'] - fact_ship['order_date']).dt.days
fact_ship['total_lead_time'] = (fact_ship['delivery_date'] - fact_ship['order_date']).dt.days

# Join geography for region
fact_ship = fact_ship.merge(
    geography[['zip', 'region', 'city']].drop_duplicates(subset='zip'),
    on='zip', how='left'
)

# Date dims
fact_ship['ship_year']  = fact_ship['ship_date'].dt.year
fact_ship['ship_month'] = fact_ship['ship_date'].dt.month
fact_ship['ship_ym']    = fact_ship['ship_date'].dt.to_period('M').astype(str)

# Delivery performance buckets
fact_ship['delivery_bucket'] = pd.cut(
    fact_ship['delivery_days'],
    bins=[-1, 2, 5, 7, 14, 999],
    labels=['1-2 days', '3-5 days', '6-7 days', '8-14 days', '15+ days']
)

save(fact_ship, 'fact_shipments_enriched.csv')

Đã lưu: data/processed/fact_shipments_enriched.csv.csv


In [54]:
print(sales.columns.tolist())

['Date', 'Revenue', 'COGS']


In [ ]:
#fact sales for sales analysis
fact_sales = sales.copy()
fact_sales.columns = ['date', 'revenue', 'cogs']
fact_sales['gross_profit']    = (fact_sales['revenue'] - fact_sales['cogs']).round(2)
fact_sales['gross_margin_pct']= (fact_sales['gross_profit'] / fact_sales['revenue'] * 100).round(2)

# Date dimensions
fact_sales['year']         = fact_sales['date'].dt.year
fact_sales['month']        = fact_sales['date'].dt.month
fact_sales['quarter']      = fact_sales['date'].dt.quarter
fact_sales['day_of_week']  = fact_sales['date'].dt.dayofweek
fact_sales['dow_name']     = fact_sales['date'].dt.day_name()
fact_sales['is_weekend']   = (fact_sales['day_of_week'] >= 5).astype(int)
fact_sales['year_month']   = fact_sales['date'].dt.to_period('M').astype(str)

# Rolling averages
fact_sales = fact_sales.sort_values('date')
fact_sales['revenue_ma7']  = fact_sales['revenue'].rolling(7, min_periods=1).mean().round(2)
fact_sales['revenue_ma30'] = fact_sales['revenue'].rolling(30, min_periods=1).mean().round(2)
fact_sales['cogs_ma7']     = fact_sales['cogs'].rolling(7, min_periods=1).mean().round(2)
fact_sales['cogs_ma30']    = fact_sales['cogs'].rolling(30, min_periods=1).mean().round(2)

# YoY growth (compare to same day last year)
fact_sales['revenue_ly'] = fact_sales['revenue'].shift(365)
fact_sales['yoy_growth_pct'] = np.where(
    fact_sales['revenue_ly'] > 0,
    ((fact_sales['revenue'] - fact_sales['revenue_ly']) / fact_sales['revenue_ly'] * 100).round(2),
    np.nan
)
fact_sales.drop(columns=['revenue_ly'], inplace=True)

save(fact_sales, 'fact_sales_daily.csv')

['Date', 'Revenue', 'COGS']
Đã lưu: data/processed/fact_sales_daily.csv.csv


In [56]:
#fact inventory for inventory analysis
fact_inv = inventory.copy()
fact_inv = fact_inv.merge(
    products[['product_id', 'price', 'cogs']].rename(
        columns={'price': 'product_price', 'cogs': 'product_cogs'}
    ),
    on='product_id', how='left'
)
fact_inv['inventory_value_retail'] = (fact_inv['stock_on_hand'] * fact_inv['product_price']).round(2)
fact_inv['inventory_value_cost']   = (fact_inv['stock_on_hand'] * fact_inv['product_cogs']).round(2)

# Inventory health label
def inv_health(row):
    if row['stockout_flag'] == 1:
        return 'Stockout'
    elif row['overstock_flag'] == 1:
        return 'Overstock'
    elif row['reorder_flag'] == 1:
        return 'Reorder Needed'
    else:
        return 'Healthy'

fact_inv['inventory_health'] = fact_inv.apply(inv_health, axis=1)

save(fact_inv, 'fact_inventory.csv')

Đã lưu: data/processed/fact_inventory.csv.csv


In [57]:
fact_web = web_traffic.copy()
fact_web.rename(columns={'date': 'traffic_date'}, inplace=True)
fact_web['year']  = fact_web['traffic_date'].dt.year
fact_web['month'] = fact_web['traffic_date'].dt.month
fact_web['year_month'] = fact_web['traffic_date'].dt.to_period('M').astype(str)

# Pages per session
fact_web['pages_per_session'] = (fact_web['page_views'] / fact_web['sessions']).round(2)

save(fact_web, 'fact_web_traffic.csv')

Đã lưu: data/processed/fact_web_traffic.csv.csv


In [69]:
# agg cohort retention analysis
first_order = orders[orders['order_status'] != 'cancelled'].groupby('customer_id')['order_date'].min().reset_index()
first_order.columns = ['customer_id', 'cohort_date']
first_order['cohort_month'] = first_order['cohort_date'].dt.to_period('M')
print(first_order.head())

# All orders with cohort
orders_cohort = orders[orders['order_status'] != 'cancelled'][['order_id', 'customer_id', 'order_date']].copy()
orders_cohort['order_month'] = orders_cohort['order_date'].dt.to_period('M')
orders_cohort = orders_cohort.merge(first_order[['customer_id', 'cohort_month']], on='customer_id')
print(orders_cohort.head())

# Period number (months since cohort)
orders_cohort['period_number'] = (
    (orders_cohort['order_month'].dt.year - orders_cohort['cohort_month'].dt.year) * 12 +
    (orders_cohort['order_month'].dt.month - orders_cohort['cohort_month'].dt.month)
)
print(orders_cohort.tail())

cohort_sizes = orders_cohort.groupby('cohort_month')['customer_id'].nunique().reset_index()
cohort_sizes.columns = ['cohort_month', 'cohort_size']
print(cohort_sizes.head())

retention = orders_cohort.groupby(['cohort_month', 'period_number'])['customer_id'].nunique().reset_index()
retention.columns = ['cohort_month', 'period_number', 'active_customers']
print(f"retention:\n{retention.head()}")

retention = retention.merge(cohort_sizes, on='cohort_month')
retention['retention_rate'] = (retention['active_customers'] / retention['cohort_size'] * 100).round(2)
print(f"Retention head:\n{retention.head()}")

retention['cohort_month'] = retention['cohort_month'].astype(str)
retention = retention[retention['period_number'] <= 24]

save(retention, 'agg_cohort_retention.csv')


   customer_id cohort_date cohort_month
0            1  2012-07-25      2012-07
1            2  2013-09-20      2013-09
2            3  2012-08-27      2012-08
3            4  2020-06-28      2020-06
4            5  2012-08-09      2012-08
   order_id  customer_id order_date order_month cohort_month
0         1        58578 2012-07-04     2012-07      2012-07
1         2        58621 2012-07-04     2012-07      2012-07
2         3        58811 2012-07-04     2012-07      2012-07
3         4        59453 2012-07-04     2012-07      2012-07
4         6        57821 2012-07-06     2012-07      2012-07
        order_id  customer_id order_date order_month cohort_month  \
587478    834372        19490 2022-12-31     2022-12      2012-07   
587479    834377        73046 2022-12-31     2022-12      2012-12   
587480    834387       107723 2022-12-31     2022-12      2013-03   
587481    834392       139431 2022-12-31     2022-12      2012-08   
587482    834397       136302 2022-12-31     2022

In [70]:

# ── 1. DOANH THU THEO THÁNG ──────────────────────────────────────────
# Gộp fact_sales theo từng tháng, tính các chỉ số doanh thu cơ bản
monthly_sales = fact_sales.groupby('year_month').agg(
    revenue       = ('revenue', 'sum'),        # tổng doanh thu tháng
    cogs          = ('cogs', 'sum'),            # tổng chi phí tháng
    gross_profit  = ('gross_profit', 'sum'),    # tổng lợi nhuận gộp
    avg_daily_rev = ('revenue', 'mean'),        # doanh thu trung bình mỗi ngày
    days_in_month = ('revenue', 'count'),       # số ngày có dữ liệu trong tháng
).reset_index()
# Biên lợi nhuận tháng đó (%) 
monthly_sales['gross_margin_pct'] = (monthly_sales['gross_profit'] / monthly_sales['revenue'] * 100).round(2)

# ── 2. ĐƠN HÀNG THEO THÁNG ───────────────────────────────────────────
orders_monthly = orders.copy()
orders_monthly['year_month'] = orders_monthly['order_date'].dt.to_period('M').astype(str)

mo = orders_monthly.groupby('year_month').agg(
    total_orders     = ('order_id', 'nunique'),                          # tổng số đơn hàng
    unique_customers = ('customer_id', 'nunique'),                       # số khách khác nhau đặt hàng
    cancelled_orders = ('order_status', lambda x: (x == 'cancelled').sum()), # đếm đơn bị huỷ
).reset_index()
# Tỷ lệ huỷ đơn (%) 
mo['cancel_rate_pct'] = (mo['cancelled_orders'] / mo['total_orders'] * 100).round(2)

# ── 3. TRẢ HÀNG THEO THÁNG ───────────────────────────────────────────
returns_monthly = returns.copy()
returns_monthly['year_month'] = returns_monthly['return_date'].dt.to_period('M').astype(str)

mr = returns_monthly.groupby('year_month').agg(
    total_returns = ('return_id', 'count'),      # số lần trả hàng
    total_refund  = ('refund_amount', 'sum'),     # tổng tiền đã hoàn lại cho khách
).reset_index()

# ── 4. GỘP CẢ 3 NGUỒN LẠI ───────────────────────────────────────────
monthly = monthly_sales.merge(mo, on='year_month', how='left')
monthly = monthly.merge(mr, on='year_month', how='left')
monthly['total_returns'] = monthly['total_returns'].fillna(0).astype(int)
monthly['total_refund']  = monthly['total_refund'].fillna(0)

# Tỷ lệ trả hàng (%) 
monthly['return_rate_pct'] = np.where(
    monthly['total_orders'] > 0,
    (monthly['total_returns'] / monthly['total_orders'] * 100).round(2),
    0
)

# Giá trị trung bình mỗi đơn hàng 
monthly['aov'] = (monthly['revenue'] / monthly['total_orders']).round(2)

# ── 5. TĂNG TRƯỞNG DOANH THU SO VỚI THÁNG TRƯỚC (MoM) ───────────────
monthly = monthly.sort_values('year_month')
monthly['revenue_prev'] = monthly['revenue'].shift(1)  

# Nếu âm liên tiếp nhiều tháng 
monthly['mom_growth_pct'] = np.where(
    monthly['revenue_prev'] > 0,
    ((monthly['revenue'] - monthly['revenue_prev']) / monthly['revenue_prev'] * 100).round(2),
    np.nan  
)
monthly.drop(columns=['revenue_prev'], inplace=True)  
save(monthly, 'agg_monthly_summary.csv')

Đã lưu: data/processed/agg_monthly_summary.csv.csv


In [71]:
reviews_enriched = reviews.merge(
    products[['product_id', 'category', 'segment']],
    on='product_id', how='left'
)
reviews_enriched['review_ym'] = reviews_enriched['review_date'].dt.to_period('M').astype(str)

rev_summary = reviews_enriched.groupby(['review_ym', 'category']).agg(
    avg_rating    = ('rating', 'mean'),
    review_count  = ('review_id', 'count'),
    pct_5_star    = ('rating', lambda x: (x == 5).mean() * 100),
    pct_1_star    = ('rating', lambda x: (x == 1).mean() * 100),
).reset_index()
rev_summary['avg_rating'] = rev_summary['avg_rating'].round(2)
rev_summary['pct_5_star'] = rev_summary['pct_5_star'].round(2)
rev_summary['pct_1_star'] = rev_summary['pct_1_star'].round(2)

save(rev_summary, 'agg_reviews_summary.csv')

Đã lưu: data/processed/agg_reviews_summary.csv.csv


## Multiple choice

In [24]:
# Q1 Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần
# mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)
orders = tables["orders"].copy()
orders["order_date"] = pd.to_datetime(orders["order_date"])

order_counts = orders.groupby("customer_id")["order_id"].count()
multi_buyers = order_counts[order_counts > 1].index
orders_multi = orders[orders["customer_id"].isin(multi_buyers)]

orders_sorted = orders_multi.sort_values(["customer_id", "order_date"])
orders_sorted["prev_date"] = orders_sorted.groupby("customer_id")["order_date"].shift(1)
orders_sorted["gap_days"]  = (orders_sorted["order_date"] - orders_sorted["prev_date"]).dt.days
gaps = orders_sorted["gap_days"].dropna()

print(f"Median inter-order gap: {gaps.median():.0f} ngày")


Median inter-order gap: 144 ngày


In [25]:
# Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
# trung bình cao nhất, với công thức (price − cogs)/price?
products = tables["products"].copy()
products["gross_margin"] = (products["price"] - products["cogs"]) / products["price"]

best_segment = products.groupby("segment")["gross_margin"].mean().idxmax()
print(f"Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất: {best_segment}")


Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất: Standard


In [26]:
# Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
# returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?
returns = tables["returns"].copy()
streetwear_returns = returns.merge(
    tables["products"][tables["products"]["category"] == "Streetwear"][["product_id"]],
    on="product_id",
    how="inner"
)
most_common_reason = streetwear_returns["return_reason"].mode()[0]
print(f"Lý do trả hàng phổ biến nhất cho Streetwear: {most_common_reason}")

Lý do trả hàng phổ biến nhất cho Streetwear: wrong_size


In [27]:
# Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
# bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?
web_traffic = tables["web_traffic"].copy()
bounce_rates = web_traffic.groupby("traffic_source")["bounce_rate"].mean()
best_source = bounce_rates.idxmin()
print(f"Traffic source có bounce rate trung bình thấp nhất: {best_source}")

Traffic source có bounce rate trung bình thấp nhất: email_campaign


In [28]:
# Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
# không null) xấp xỉ là bao nhiêu?
promo_rate = tables["order_items"]["promo_id"].notna().mean() * 100
print(f"Tỷ lệ đơn hàng có khuyến mãi: {promo_rate:.1f}%")

Tỷ lệ đơn hàng có khuyến mãi: 38.7%


In [29]:
# Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
# đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
# nhóm)
customers = tables["customers"].copy()
customers = customers[customers["age_group"].notna()]

order_counts = tables["orders"].groupby("customer_id")["order_id"].count().reset_index()
order_counts.columns = ["customer_id", "order_count"]
customers = customers.merge(order_counts, on="customer_id", how="left")
age_group_counts = customers.groupby("age_group")["customer_id"].count()
age_group_orders = customers.groupby("age_group")["order_count"].sum()
avg_orders_per_customer = age_group_orders / age_group_counts
best_age_group = avg_orders_per_customer.idxmax()
print(f"Nhóm tuổi có số đơn hàng trung bình cao nhất: {best_age_group} voi {avg_orders_per_customer[best_age_group]:.2f} đơn hàng mỗi khách hàng")

Nhóm tuổi có số đơn hàng trung bình cao nhất: 55+ voi 5.41 đơn hàng mỗi khách hàng


In [ ]:
# Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
# sales_train.csv?



ValueError: You are trying to merge on datetime64[ns] and int64 columns for key 'date'. If you wish to proceed you should use pd.concat

In [31]:
# Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
# thanh toán nào được sử dụng nhiều nhất?
orders = tables["orders"].copy()
cancelled_orders = orders[orders["order_status"] == "cancelled"]
most_common_payment = cancelled_orders["payment_method"].mode()[0]
print(f"Phương thức thanh toán phổ biến nhất trong đơn hàng bị hủy: {most_common_payment}")

Phương thức thanh toán phổ biến nhất trong đơn hàng bị hủy: credit_card


In [32]:

# Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
# nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
# với products theo product_id)?
returns = tables["returns"].copy()
products = tables["products"].copy()
order_items = tables["order_items"].copy()

order_items = order_items.merge(
    products[["product_id", "size"]],
    on="product_id",
    how="left"
)

returns = returns.merge(
    products[["product_id", "size"]],
    on="product_id",
    how="left"
)
size_counts = order_items["size"].value_counts()
size_returns = returns["size"].value_counts()
return_rates = size_returns / size_counts
best_size = return_rates.idxmax()
print(f"Kích thước có tỷ lệ trả hàng cao nhất: {best_size} với tỷ lệ {return_rates[best_size]:.2%}")

Kích thước có tỷ lệ trả hàng cao nhất: S với tỷ lệ 5.65%


In [33]:
# Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
# mỗi đơn hàng cao nhất?
payments = tables["payments"]

avg_payment = payments.groupby("installments")["payment_value"].mean()

best_plan = avg_payment.idxmax()

print(best_plan)
print(avg_payment[best_plan])

6
24446.65440296606
